# Day 3: Python + Docker (Production Mindset) 🚀

## Goal

Move the FastAPI app from a flat script or notebook into a standard project folder structure and containerize it.

By the end of this notebook you will have:

- an organized Python backend structure
- a `Dockerfile` to containerize the API
- an understanding of how to build and run the container
- a runnable Docker container serving the API

This prepares the app for real-world deployment.

## ❌ Bad Folder Structure

Flat, unorganized projects become hard to scale and maintain.

```
main.py
utils.py
random.py
test.py
```

**Problems:**
- No separation of concerns
- Hard to scale in teams
- Difficult to maintain

## ✅ Good Folder Structure

A clean, scalable structure:

```
api_project/
├── app/
│   ├── main.py        # entry point
│   └── api/
│       └── endpoints.py
└── requirements.txt
```

**Why this works:**
- Clear separation
- Easy to scale
- Predictable layout

In [ ]:
import os

base_dir = "api_project"
os.makedirs(f"{base_dir}/app/api", exist_ok=True)

open(f"{base_dir}/app/__init__.py","w").close()
open(f"{base_dir}/app/api/__init__.py","w").close()

print("Structure ready")

## 🔥 The 'Works on My Machine' Problem

Even with good structure, apps break across environments.

- Different Python versions
- Missing dependencies
- System-level issues

In [ ]:
import sys
print("Running Python:", sys.version)

## 🐳 What is Docker?

Docker is a platform that packages your application with:
- Code
- Runtime (Python)
- Dependencies
- System libraries

👉 Think: **Portable environment that runs anywhere**

## 🚀 Why Docker?

Without Docker:
- Version mismatch
- Dependency issues
- Manual setup errors

With Docker:
- Same environment everywhere
- Easy deployment
- No setup headaches

## 🧠 Why Docker Improves Python Reliability

Python apps often break due to:
- virtualenv inconsistencies
- system dependencies
- OS differences

Docker fixes this by locking everything.

## 🧱 Folder Structure with Docker

```
api_project/
├── app/
├── requirements.txt
├── Dockerfile
├── docker-compose.yml
```

## 💻 Writing Application Code

In [ ]:
%%writefile api_project/app/api/endpoints.py
from fastapi import APIRouter

router = APIRouter()

@router.get("/health")
def health():
    return {"status": "ok"}

In [ ]:
%%writefile api_project/app/main.py
from fastapi import FastAPI
from app.api.endpoints import router

app = FastAPI()
app.include_router(router, prefix="/api/v1")

In [ ]:
%%writefile api_project/requirements.txt
fastapi
uvicorn

## ⚙️ What is a Dockerfile?

A Dockerfile is a **step-by-step recipe to set up your app on a fresh machine**.

Each line represents one setup step.

In [ ]:
%%writefile api_project/Dockerfile
# 1. Base image (Python installed)
FROM python:3.12-slim

# 2. Set working directory
WORKDIR /src

# 3. Copy dependencies
COPY requirements.txt .

# 4. Install dependencies
RUN pip install --no-cache-dir -r requirements.txt

# 5. Copy application code
COPY ./app ./app

# 6. Expose port
EXPOSE 8000

# 7. Run application
CMD ["uvicorn","app.main:app","--host","0.0.0.0","--port","8000"]

## ⚙️ What is Docker Compose?

Docker Compose is used to **run and manage containers easily**.

👉 Instead of long commands, define everything once.

In [ ]:
%%writefile api_project/docker-compose.yml
version: '3'
services:
  api:
    build: .
    ports:
      - "8000:8000"


👉 **Dockerfile = build image | Docker Compose = run container**

## 💥 Failure Scenario

In [ ]:
%%writefile api_project/FailDockerfile
FROM python:3.12-slim
WORKDIR /src

COPY requirements.txt .
RUN pip install -r requirements.txt

# Missing COPY ./app ./app

CMD ["uvicorn","app.main:app","--host","0.0.0.0","--port","8000"]

In [ ]:
!cd api_project && docker build -f FailDockerfile -t bad-api .

In [ ]:
!docker run --rm bad-api

👉 Error: module not found → because app code wasn't copied

## 🚀 Build, Run & Test

In [ ]:
!cd api_project && docker build -t good-api .

In [ ]:
!docker run -d -p 8000:8000 --name myapi good-api

In [ ]:
!curl http://127.0.0.1:8000/api/v1/health

In [ ]:
# To build the project
docker-compose build


# To start the docker containers
docker-compose up -d

# To stop the docker containers
docker-compose down

## 🧪 Logs

In [ ]:
!docker logs myapi

## 🏭 Production Tips

In [ ]:
%%writefile api_project/.dockerignore
__pycache__/
*.pyc
.env

- Use workers
- Reduce image size
- Avoid copying unnecessary files

## ⚠️ Common Mistakes

- Missing port mapping
- Using localhost instead of 0.0.0.0
- Not rebuilding image

In [ ]:
!docker stop myapi
!docker rm myapi

## 🧠 Recap

You have successfully:

- Restructured a basic Python app into a standard `app/` structure with separate routers.
- Written a `Dockerfile` that packages Python dependencies and source code.
- Built a standard runnable container (`day3-api:latest`).
- Validated the application running fully isolated inside Docker.

This stable deployment pattern will be used when we deploy more complex AI endpoints in subsequent days.